In [13]:
import sys
import importlib
from pathlib import Path

import numpy as np

# Find the repository root robustly from the notebook working directory
cwd = Path.cwd().resolve()
repo_root = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "utilities" / "functions.py").exists():
        repo_root = candidate
        break

if repo_root is None:
    raise FileNotFoundError("Could not find the repository root containing utilities/functions.py")

sys.path.append(str(repo_root))  # make the repo importable from notebooks in subfolders

import utilities.functions as functions
import utilities.plot as plot

# Reload modules to reflect any changes
importlib.reload(functions)
importlib.reload(plot)

IN_start_index = 1
IN_end_index = 263
PR_start_index = 1
PR_end_index = 99
RT_start_index = 39
RT_end_index = 226

data_root = repo_root / "ms0_5"

IN_seq_path = str(data_root / "IN" / "data" / "in.reduce4.seq")
PR_seq_path = str(data_root / "PR" / "data" / "pr.exper.reduce4.seq")
RT_seq_path = str(data_root / "RT" / "data" / "rt.reduce4.seq")

IN_all_seq = functions.read_seq(IN_seq_path)
PR_all_seq = functions.read_seq(PR_seq_path)
RT_all_seq = functions.read_seq(RT_seq_path)

IN_consensus = str(data_root / "IN" / "data" / "in.consensus.reduce4.seq")
with open(IN_consensus, "r") as f:
    IN_consensus_seq = f.read().strip()

PR_consensus = str(data_root / "PR" / "data" / "pr.consensus.reduce4.seq")
with open(PR_consensus, "r") as f:
    PR_consensus_seq = f.read().strip()

RT_consensus = str(data_root / "RT" / "data" / "rt.consensus.reduce4.seq")
with open(RT_consensus, "r") as f:
    RT_consensus_seq = f.read().strip()

IN_redux = functions.get_redu_dict(str(data_root / "IN" / "data" / "in.reduce4.redux"), 1)
PR_redux = functions.get_redu_dict(str(data_root / "PR" / "data" / "pr.reduce4.redux"), 0)
RT_redux = functions.get_redu_dict(str(data_root / "RT" / "data" / "rt.reduce4.redux"), 0)

alphabet = ["A", "B", "C", "D"]


def build_J_matrix(j_file, min_position, max_position):
    """Dense coupling tensor, shape (L, L, 4, 5), indexed [p1, p2, aa_at_p1, aa_at_p2].

    Same content as functions.load_J_dict but as an array instead of a ~1M-entry
    dict, so couplings can be gathered with numpy instead of per-key lookups.
    The 5th slot on the last axis is an all-zero column standing in for
    out-of-alphabet characters (mirrors the J_dict.get(..., 0) default).
    """
    J = np.load(j_file).astype(np.float32)
    L = max_position - min_position + 1
    Jm = np.zeros((L, L, 4, 5), dtype=np.float32)
    # load_J_dict walks pos1 ascending, pos2 from pos1+1 ascending -> row-major upper triangle
    iu0, iu1 = np.triu_indices(L, 1)
    blocks = J.reshape(-1, 4, 4)
    assert blocks.shape[0] == iu0.size, "J.npy row count does not match the position range"
    Jm[iu0, iu1, :, :4] = blocks
    Jm[iu1, iu0, :, :4] = blocks.transpose(0, 2, 1)
    return Jm


IN_J = build_J_matrix(str(data_root / "IN" / "data" / "J.npy"), IN_start_index, IN_end_index)
PR_J = build_J_matrix(str(data_root / "PR" / "data" / "J_PR.npy"), PR_start_index, PR_end_index)
RT_J = build_J_matrix(str(data_root / "RT" / "data" / "J_RT.npy"), RT_start_index, RT_end_index)

IN_all_seq_unreduced = functions.read_seq(str(data_root / "IN" / "data" / "in.fullseq"))
PR_all_seq_unreduced = functions.read_seq(str(data_root / "PR" / "data" / "pr.exper.fullseq"))
RT_all_seq_unreduced = functions.read_seq(str(data_root / "RT" / "data" / "rt.fullseq"))

In [14]:
import csv

import numpy as np

# ---------------------------------------------------------------------------
# Vectorized re-implementation of the per-sequence loop.
#
# For a pair (p1: wt1->mt1, p2: wt2->mt2), calculate_dde_v2 first rewrites the
# sequence to wild type at both positions, so every quantity depends on the
# sequence only through the "background" couplings
#
#     S(p, x) = sum_{o not in {p1, p2}} J[p, o, x, seq[o]]
#
# plus the direct term J[p1, p2, ., .]. With
#     M[a1, a2] = S(p1, a1) + S(p2, a2) + J[p1, p2, a1, a2]
# we get exactly what the scalar functions compute:
#     dE1  = M[wt1, wt2] - M[mt1, wt2]
#     dE2  = M[wt1, wt2] - M[wt1, mt2]
#     dE12 = M[wt1, wt2] - M[mt1, mt2]
#     p_DMC = exp(dE12) / sum_{a1,a2} exp(M[wt1, wt2] - M[a1, a2])   (a softmax over -M)
# so the 8 numbers S(p1, ABCD) / S(p2, ABCD) are all that is needed per sequence,
# and they come from a single matmul against a one-hot encoding of the alignment.
# ---------------------------------------------------------------------------

_AA_CODE = np.full(256, 4, dtype=np.uint8)  # anything outside ABCD -> the zero-coupling slot
for _i, _c in enumerate("ABCD"):
    _AA_CODE[ord(_c)] = _i


def encode_seqs(seq_list, min_pos, max_pos):
    """One-hot encode an alignment as (N, L*5) float32, column order (position, aa)."""
    L = max_pos - min_pos + 1
    N = len(seq_list)
    raw = np.frombuffer("".join(seq_list).encode(), dtype=np.uint8)
    assert raw.size == N * L, "all sequences must span exactly min_pos..max_pos"
    codes = _AA_CODE[raw.reshape(N, L)]
    onehot = np.zeros((N * L, 5), dtype=np.float32)
    onehot[np.arange(N * L), codes.ravel()] = 1.0
    return codes, onehot.reshape(N, L * 5)


def _site_energies(onehot, Jm, p, excluded):
    """S(p, x) for x in ABCD, for every sequence -> (N, 4)."""
    A = Jm[p].copy()
    A[list(excluded)] = 0.0  # drop the two mutated positions from the background sum
    return np.asarray(onehot @ A.transpose(0, 2, 1).reshape(-1, 4), dtype=np.float64)


def pair_energies(onehot, Jm, p1i, p2i, wt1, mt1, wt2, mt2):
    """dE1, dE2, dE12 and p(DMC) for every sequence in the alignment."""
    S1 = _site_energies(onehot, Jm, p1i, (p1i, p2i))
    S2 = _site_energies(onehot, Jm, p2i, (p1i, p2i))
    J12 = Jm[p1i, p2i, :, :4].astype(np.float64)

    M = S1[:, :, None] + S2[:, None, :] + J12[None, :, :]
    base = M[:, wt1, wt2]
    de1 = base - M[:, mt1, wt2]
    de2 = base - M[:, wt1, mt2]
    de12 = base - M[:, mt1, mt2]

    # softmax over -M (the +base cancels); done shifted, so no exp overflow
    Z = -M.reshape(S1.shape[0], 16)
    Z = Z - Z.max(axis=1, keepdims=True)
    E = np.exp(Z)
    p_dmc = E[:, mt1 * 4 + mt2] / E.sum(axis=1)
    return de1, de2, de12, p_dmc


def _cat_stats(mask, p, weights, dmc, len_all_seqs, weights_sum):
    """Counts / average p / observed f for one epistasis subcategory."""
    n = int(mask.sum())
    n_dmc = int(np.count_nonzero(mask & dmc))
    p_sum = float(p[mask].sum())
    w_sub = float(weights[mask].sum())
    wp_sum = float(np.dot(p[mask], weights[mask]))
    w_dmc = float(weights[mask & dmc].sum())
    return {
        'n': n,
        'n_dmc': n_dmc,
        'w_sum': w_sub,
        'w_dmc': w_dmc,
        'avg_p_total': p_sum / len_all_seqs if len_all_seqs else 0,
        'avg_p_sub': p_sum / n if n else 0,
        'w_p_total': wp_sum / weights_sum if n else 0,
        'w_p_sub': wp_sum / w_sub if n else 0,
        'obs_f_total': n_dmc / len_all_seqs,
        'obs_f_sub': n_dmc / n if n else 0,
        'w_obs_f_total': w_dmc / weights_sum if n else 0,
        'w_obs_f_sub': w_dmc / w_sub if n else 0,
    }


def output_probs(prefix, min_pos, max_pos, all_seq, consensus_seq, redux, pairs, weights_path, J, output_csv):
    """
    Process epistasis data for a given prefix (e.g., IN, PR, RT).

    Parameters:
        prefix (str): Prefix for the dataset (e.g., 'IN', 'PR', 'RT').
        all_seq (list): List of all sequences.
        consensus_seq (str): Consensus sequence.
        redux (dict): Reduction dictionary.
        pairs (list): List of mutation pairs.
        weights_path (str): Path to the weights file.
        J (np.ndarray): Coupling tensor from build_J_matrix.
        output_csv (str): Output CSV file name.
    """
    len_all_seqs = len(all_seq)

    # Read weights from the file
    with open(weights_path, 'r') as f:
        weights = [float(line.strip()) for line in f]

    # Ensure the weights list matches the all_seq list
    assert len(weights) == len(all_seq), "Weights and sequences must have the same length."

    weights = np.asarray(weights, dtype=np.float64)
    weights_sum = float(weights.sum())

    # One-hot encoding is shared by every pair -> build it once
    codes, onehot = encode_seqs(all_seq, min_pos, max_pos)
    _, consensus_onehot = encode_seqs([consensus_seq], min_pos, max_pos)

    csv_total_data = []
    csv_data = []
    csv_weighted_data = []

    for pair in pairs:
        print(f"Processing pair: {pair}")
        pair1, pair2 = functions.split_pairs(pair)
        p1_reduced = functions.unreduced_to_reduced(redux, pair1)
        p2_reduced = functions.unreduced_to_reduced(redux, pair2)
        wt1, pos1, mt1 = functions.split_pair(p1_reduced)
        wt2, pos2, mt2 = functions.split_pair(p2_reduced)

        p1i, p2i = pos1 - min_pos, pos2 - min_pos
        iwt1, imt1 = "ABCD".index(wt1), "ABCD".index(mt1)
        iwt2, imt2 = "ABCD".index(wt2), "ABCD".index(mt2)
        idx = (p1i, p2i, iwt1, imt1, iwt2, imt2)

        # consensus reference for the flip test
        c_de1, c_de2, _, _ = pair_energies(consensus_onehot, J, *idx)
        consensus_de_diff = float(c_de1[0] - c_de2[0])

        de1, de2, de12, p_SH = pair_energies(onehot, J, *idx)

        # sequences carrying the double mutant combination
        dmc = (codes[:, p1i] == imt1) & (codes[:, p2i] == imt2)

        # ---- epistasis subcategories (same branch order as the scalar version) ----
        de_diff = de1 - de2
        flip = consensus_de_diff * de_diff < 0
        non_flip = consensus_de_diff * de_diff > 0

        both_below = (de1 < de12) & (de2 < de12)
        gof = both_below & (de12 > 0)
        rescue = both_below & ~(de12 > 0)
        comp = ~both_below & ((de1 < de12) | (de2 < de12))
        noncomp = ~both_below & ~((de1 < de12) | (de2 < de12))

        stats = {name: _cat_stats(mask, p_SH, weights, dmc, len_all_seqs, weights_sum)
                 for name, mask in (('gof', gof), ('rescue', rescue), ('comp', comp),
                                    ('noncomp', noncomp), ('flip', flip), ('non_flip', non_flip))}

        # ---- totals over all sequences ----
        total_counts = len_all_seqs
        total_with_DMC_count = int(np.count_nonzero(dmc))
        average_p_all = float(p_SH.sum()) / len_all_seqs if len_all_seqs else 0
        weighted_all_prob = float(np.dot(p_SH, weights)) / weights_sum if len_all_seqs else 0
        weights_with_DMC_sum = float(weights[dmc].sum())
        actual_p_all = total_with_DMC_count / total_counts if total_counts > 0 else 0
        actual_p_all_weighted = weights_with_DMC_sum / weights_sum if len_all_seqs else 0

        # Append data
        csv_total_data.append([pair, 'Total', total_counts, total_with_DMC_count, average_p_all, actual_p_all, average_p_all, actual_p_all, weights_sum, weights_with_DMC_sum, weighted_all_prob, actual_p_all_weighted, weighted_all_prob, actual_p_all_weighted])

        # Append data for each epistasis subset
        # non-weighted
        csv_data.append(['total'])
        for label, key in (('Gain_of_function', 'gof'), ('rescue', 'rescue'),
                           ('compensatory', 'comp'), ('non_compensatory', 'noncomp')):
            s = stats[key]
            csv_data.append([pair, label, s['n'], s['n_dmc'], s['avg_p_total'], s['obs_f_total'], s['avg_p_sub'], s['obs_f_sub']])
        csv_data.append([])
        for label, key in (('flipped', 'flip'), ('non_flipped', 'non_flip')):
            s = stats[key]
            csv_data.append([pair, label, s['n'], s['n_dmc'], s['avg_p_total'], s['obs_f_total'], s['avg_p_sub'], s['obs_f_sub']])
        csv_data.append([])

        # weighted
        csv_weighted_data.append(['total'])
        for label, key in (('Gain_of_function', 'gof'), ('rescue', 'rescue'),
                           ('compensatory', 'comp'), ('non_compensatory', 'noncomp')):
            s = stats[key]
            csv_weighted_data.append([pair, label, s['w_sum'], s['w_dmc'], s['w_p_total'], s['w_obs_f_total'], s['w_p_sub'], s['w_obs_f_sub']])
        csv_weighted_data.append([])
        s = stats['flip']
        csv_weighted_data.append([pair, 'flipped', s['w_sum'], s['w_dmc'], s['w_p_total'], s['w_obs_f_total'], s['w_p_sub'], s['w_obs_f_sub']])
        s = stats['non_flip']
        # NOTE: last column repeats w_p_sub, as in the original implementation
        csv_weighted_data.append([pair, 'non_flipped', s['w_sum'], s['w_dmc'], s['w_p_total'], s['w_obs_f_total'], s['w_p_sub'], s['w_p_sub']])
        csv_weighted_data.append([])

        with open(output_csv, 'w', newline='') as csvfile:
            csv_writer = csv.writer(csvfile)
            csv_writer.writerow(['mutation_pair', 'epistasis_subset', 'num_seqs', 'num_with_DMC','average_p_total', 'observed_f_total','average_p_subcategory', 'observed_f_subcategory','', 'weights_sum','weights_sum_with_DMC','weighted_average_p_total', 'weighted_observed_f_total', 'weighted_average_p_subcategory', 'weighted_observed_f_subcategory'])
            count = 0
            for row, weighted_row in zip(csv_data, csv_weighted_data):
                if row == [] or weighted_row == []:
                    csv_writer.writerow([])
                elif row == ['total'] or weighted_row == ['total']:
                    csv_writer.writerow([f"{csv_total_data[count][0]}", f"{csv_total_data[count][1]}", f"{csv_total_data[count][2]}", f"{csv_total_data[count][3]:}", f"{csv_total_data[count][4]:.4f}", f"{csv_total_data[count][5]:.4f}", f"{csv_total_data[count][6]:.4f}",f"{csv_total_data[count][7]:.4f}", '', f"{csv_total_data[count][8]:.4f}", f"{csv_total_data[count][9]:.4f}", f"{csv_total_data[count][10]:.4f}", f"{csv_total_data[count][11]:.4f}", f"{csv_total_data[count][12]:.4f}", f"{csv_total_data[count][13]:.4f}"])
                    count += 1
                else:
                    csv_writer.writerow([f"{row[0]}", f"{row[1]}", f"{row[2]}", f"{row[3]}", f"{row[4]:.4f}", f"{row[5]:.4f}", f"{row[6]:.4f}", f"{row[7]:.4f}", '', f"{weighted_row[2]:.4f}", f"{weighted_row[3]:.4f}", f"{weighted_row[4]:.4f}", f"{weighted_row[5]:.4f}", f"{weighted_row[6]:.4f}", f"{weighted_row[7]:.4f}"])
            # Append the total data for each mutation pair
    print(f"CSV file {output_csv} created successfully.")

In [15]:
IN_weights_path = 'IN/data/in.weights.txt'
len_IN_all_seqs = len(IN_all_seq)
IN_weights_path = str(data_root / "IN" / "data" / "in.weights.txt")
with open(IN_weights_path, "r") as f:
    IN_weights = [float(line.strip()) for line in f]

# Ensure the weights list matches the IN_all_seq list
assert len(IN_weights) == len_IN_all_seqs, "Weights and sequences must have the same length."

IN_pairs = [
    # 'G140S-Q148H',
    # 'Y143C-S230R',
    # 'G140A-Q148K',
    # 'G140S-Q148R',
    # 'G140S-Q148K',
    # 'G140A-Q148R',
    # 'E138K-Q148K',
    # 'G140A-Q148H',
    # 'E138K-Q148R',
    # 'Y143C-S230K',
    # 'N155H-E170A',
    # 'E138K-S147G',
    # 'S147G-Q148R',
    # 'Y143R-I151V',
    # 'E138K-Q148H',
    # 'S147G-L158V',
    # 'E92Q-K215R',
    # 'E138A-Q148H',
    # 'E138K-S230K',
    # 'E138K-Y194C',
    'G140S-Q148H',
    'Y143C-S230R',
    'E157Q-K160Q',
    'S119G-T122I',
    'G140S-Q148R',
    'K219N-N222K',
    'E138K-Q148R',
    'E11D-S195T',
    'T125A-V126L',
    'S119P-T122I',
    'Q221S-N222K',
    'L28I-V37I',
    'T97A-Y143R',
    'E11D-S24N',
    'E138K-S147G',
    'D6E-E10D',
    'T97A-S119R',
    'S255N-D256E',
    'I220L-Y227F',
    'E11D-A21T',
]
output_probs('IN', 1,263,IN_all_seq, IN_consensus_seq, IN_redux, IN_pairs, IN_weights_path, IN_J, 'integrase_all_probabilities_v14.csv')

Processing pair: G140S-Q148H
Processing pair: Y143C-S230R
Processing pair: E157Q-K160Q
Processing pair: S119G-T122I
Processing pair: G140S-Q148R
Processing pair: K219N-N222K
Processing pair: E138K-Q148R
Processing pair: E11D-S195T
Processing pair: T125A-V126L
Processing pair: S119P-T122I
Processing pair: Q221S-N222K
Processing pair: L28I-V37I
Processing pair: T97A-Y143R
Processing pair: E11D-S24N
Processing pair: E138K-S147G
Processing pair: D6E-E10D
Processing pair: T97A-S119R
Processing pair: S255N-D256E
Processing pair: I220L-Y227F
Processing pair: E11D-A21T
CSV file integrase_all_probabilities_v14.csv created successfully.


In [16]:
PR_weights_path = str(data_root / "PR" / "data" / "pr.exper.weights.txt")
len_PR_all_seqs = len(PR_all_seq)
with open(PR_weights_path, "r") as f:
    PR_weights = [float(line.strip()) for line in f]

# Ensure the weights list matches the IN_all_seq list
assert len(PR_weights) == len_PR_all_seqs, "Weights and sequences must have the same length."

PR_pairs = [
    # 'D30N-N88D',
    # 'V32I-I47V',
    # 'G48V-I54A',
    # 'D30N-K45Q',
    # 'I54A-V82A',
    # 'I54V-V82A',
    # 'M46I-L76V',
    # 'I54V-V82T',
    # 'I54A-V82T',
    # 'G48V-V82A',
    # 'I54A-A71I',
    # 'M46I-N88T',
    # 'L90M-C95F',
    # 'V32I-M46I',
    # 'G48V-V82T',
    # 'I54V-T91S',
    # 'M46I-F53Y',
    # 'M46L-K55R',
    # 'M46L-V82A',
    # 'M46I-K55R',
    'D30N-N88D',
    'V32I-I47V',
    'K20R-M36I',
    'G16E-P39S',
    'I54A-V82A',
    'I54V-V82A',
    'M46I-L76V',
    'T12P-K14R',
    'L33F-I54L',
    'I54V-V82T',
    'G73T-L90M',
    'G48V-V82A',
    'G73S-L90M',
    'R57K-Q61N',
    'D60E-Q61E',
    'M36L-I62V',
    'L10I-I54A',
    'L10F-I84V',
    'P79A-I84V',
    'T12S-L19I',
]
output_probs('PR', 1,99,PR_all_seq, PR_consensus_seq, PR_redux, PR_pairs, PR_weights_path, PR_J, 'protease_all_probabilities_v14.csv')

Processing pair: D30N-N88D


Processing pair: V32I-I47V
Processing pair: K20R-M36I
Processing pair: G16E-P39S
Processing pair: I54A-V82A
Processing pair: I54V-V82A
Processing pair: M46I-L76V
Processing pair: T12P-K14R
Processing pair: L33F-I54L
Processing pair: I54V-V82T
Processing pair: G73T-L90M
Processing pair: G48V-V82A
Processing pair: G73S-L90M
Processing pair: R57K-Q61N
Processing pair: D60E-Q61E
Processing pair: M36L-I62V
Processing pair: L10I-I54A
Processing pair: L10F-I84V
Processing pair: P79A-I84V
Processing pair: T12S-L19I
CSV file protease_all_probabilities_v14.csv created successfully.


In [17]:
# RT_weights_path = 'RT/data/rt.weights.txt'
# len_RT_all_seqs = len(RT_all_seq)
# with open(RT_weights_path, 'r') as f:
#     RT_weights = [float(line.strip()) for line in f]

# # Ensure the weights list matches the IN_all_seq list
# assert len(RT_weights) == len_RT_all_seqs, "Weights and sequences must have the same length."

# RT_pairs = [
#     # 'K101E-G190S',
#     # 'K101E-G190A',
#     # 'K103N-P225H',
#     # 'L100I-K103N',
#     # 'K101P-K103S',
#     # 'Y181C-H221Y',
#     # 'K103S-G190A',
#     # 'K103S-P225H',
#     # 'L100I-K103R',
#     # 'V108I-H221Y',
#     # 'K103S-D192N',
#     # 'L100I-K103S',
#     # 'K101E-E138A',
#     # 'Y181C-G190A',
#     # 'K103S-D177N',
#     # 'V108I-V189I',
#     # 'K101E-E138K',
#     # 'E138A-G190E',
#     # 'K101P-D192N',
#     # 'V108I-L109V',

# ]

# output_probs('NNRTI', 39,226,RT_all_seq, RT_consensus_seq, RT_redux, RT_pairs, RT_weights_path, RT_J, 'reverseTranscriptase_NNRTI_probabilities_v14.csv')

In [18]:
RT_weights_path = str(data_root / "RT" / "data" / "rt.weights.txt")
len_RT_all_seqs = len(RT_all_seq)
with open(RT_weights_path, "r") as f:
    RT_weights = [float(line.strip()) for line in f]

# Ensure the weights list matches the IN_all_seq list
assert len(RT_weights) == len_RT_all_seqs, "Weights and sequences must have the same length."

RT_pairs = [
    # 'F116Y-Q151M',
    # 'M41L-T215Y',
    # 'V75I-I132L',
    # 'K70R-K219E',
    # 'D67N-K219Q',
    # 'K70R-K219Q',
    # 'L210W-T215Y',
    # 'F116Y-Q151L',
    # 'K65R-S68N',
    # 'V75I-F77L',
    # 'M41L-T215F',
    # 'D67N-K219E',
    # 'L210W-T215S',
    # 'M41L-T215S',
    # 'D67N-K70R',
    # 'L74V-Y115F',
    # 'L74V-L100I',
    # 'A62V-V75I',
    # 'F116Y-Q151R',
    # 'A62V-V75T',
    'F116Y-Q151M',
    'M41L-T215Y',
    'V75M-F77L',
    'K101E-G190S',
    'E203K-K223E',
    'K43E-E44A',
    'K70R-K219E',
    'D67N-K219Q',
    'K70R-K219Q',
    'L210W-T215Y',
    'K103R-V179D',
    'K101E-G190A',
    'K103N-P225H',
    'L100I-K103N',
    'V75I-F77L',
    'M41L-T215F',
    'D67G-K219E',
    'D121Y-K122E',
    'D67N-K219E',
    'D121H-K122E',
]
output_probs('NRTI', 39,226,RT_all_seq, RT_consensus_seq, RT_redux, RT_pairs, RT_weights_path, RT_J, 'reverseTranscriptase_both_probabilities_v14.csv')

Processing pair: F116Y-Q151M
Processing pair: M41L-T215Y
Processing pair: V75M-F77L
Processing pair: K101E-G190S
Processing pair: E203K-K223E
Processing pair: K43E-E44A
Processing pair: K70R-K219E
Processing pair: D67N-K219Q
Processing pair: K70R-K219Q
Processing pair: L210W-T215Y
Processing pair: K103R-V179D
Processing pair: K101E-G190A
Processing pair: K103N-P225H
Processing pair: L100I-K103N
Processing pair: V75I-F77L
Processing pair: M41L-T215F
Processing pair: D67G-K219E
Processing pair: D121Y-K122E
Processing pair: D67N-K219E
Processing pair: D121H-K122E
CSV file reverseTranscriptase_both_probabilities_v14.csv created successfully.
